In [ ]:
import pandas as pd
import numpy as np
import os
df = pd.DataFrame()
df = pd.read_csv(r'data/diabetic_data.csv')

In [ ]:
import sqlite3
EW = pd.ExcelWriter('readmission_by_col.xlsx')
new_connection = sqlite3.connect('data/diabetic_data.db')

cols = df.columns.tolist()
for col in cols:
    query = f"""
    WITH readmitted_counts AS (
        SELECT COUNT(readmitted) AS count, `{col}`
        FROM diabetic_data
        GROUP BY `{col}`
        ),
        thirty_days_readmitted AS (
        SELECT COUNT(*) AS count, `{col}`
        FROM diabetic_data
        WHERE readmitted = '<30'
        GROUP BY `{col}`
        )
    
    SELECT
        (thirty_days_readmitted.count*100.0/readmitted_counts.count), thirty_days_readmitted.`{col}`, readmitted_counts.count
        FROM thirty_days_readmitted
        JOIN readmitted_counts ON thirty_days_readmitted.`{col}` = readmitted_counts.`{col}`
    """
    result = pd.read_sql_query(query, new_connection)
    result.to_excel(EW, sheet_name=col, index=False)
EW.close()
new_connection.close()